# **CV融合算子开发**
本章以 Matmul + LeakyReLU 为例，依次学习：
- 环境准备与 CV 融合算子分析
- 基于高阶 API 的融合实现：Host Tiling、Matmul API、编译测试
- 基于基础 API 的融合实现：显式内存分配、数据搬运、矩阵计算与核间同步
- 本章小结
- 课后实践：MatmulAbs 融合算子

## **1. 环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及编译。

In [ ]:
!mkdir -p Sources/03.05

import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

---
## **2. CV融合算子分析**
假设我们有个简单的小模型，里面包含了一个Matmul、一个Leakyrelu算子，原始的模型如下：  

<img src="./images/03.05/cv_operator_model.png">

### **2.1 算子输入输出与Shape信息**
融合后的算子仅包含输入 `a`、`b`、`bias` 和输出 `c`，其计算流程为：

$$
c = leakyrelu(a \times b + bias, alpha)
$$

其中 `a`、`b` 为 `half` 类型输入，`bias` 为 `float` 类型输入，`c` 为 `float` 类型输出。此处仅以 `M = 1024`，`K = 256`，`N = 640` 输入数据作为演示示例，在真实业务场景下，可灵活调整输入数据的shape参数，也可直接支持泛化输入形式。

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 650px; border: 1px solid #ddd;">
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子名</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" >matmul_leakyrelu_custom</td>
  </tr>
  <tr>
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(M, K) / (1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(K, N) / (256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(N) / (640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(M, N) / (1024, 640)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>


### **2.2 执行步骤拆解**
Matmul与LeakyReLU两个算子原生采用串行执行模式，完整流程可拆解为 **6 个独立且串行的步骤**：
- 阶段1（Matmul执行）：Matmul输入数据搬入 → Cube核执行Matmul计算 → Matmul输出数据搬出  

- 阶段2（LeakyReLU执行）：LeakyReLU输入数据搬入 → Vector核执行 LeakyReLU计算 → LeakyReLU输出数据搬出

### **2.3 核心效率瓶颈**
- **计算核资源利用率低**：Matmul依赖Cube核运算，LeakyReLU依赖Vector核运算，串行执行时总有一类计算核处于完全空闲状态；  

- **数据传输开销大**：LeakyReLU输入完全依赖Matmul输出，中间结果需经历“全局内存 ↔ 计算核”的两次完整搬移，额外增加访存耗时；  

- **执行链路冗长**：6 个独立步骤串行执行，无并行优化空间，端到端耗时高。  


将Matmul与LeakyReLU融合为单一算子后，通过「数据切块+核间协同+本地缓存」重构执行流程，整体以 **5 个核心步骤循环执行** 完成全量数据运算：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 1485px; border: 1px solid #ddd;">
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">步骤</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">操作内容</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">核心目的</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">1</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">数据切块：将Matmul输入数据（a矩阵、b矩阵、bias偏置）按 Cube 核单次计算能力切分为若干数据块</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube每运算完成一小块数据可以直接让Vector核开始计算，避免Vector核空闲等待</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">2</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube核数据搬入：将单块输入数据直接搬入Cube核本地存储</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">数据需要放在Cube核上完成矩阵乘运算</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">3</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube核并行计算Matmul：各Cube核并行完成对应数据块的Matmul计算，结果暂存于LocalTensor</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">利用Matmul API的能力，直接输出Matmul计算结果到UB，减少编写搬运代码</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">4</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Vector核并行计算LeakyReLU：直接读取LocalTensor中的Matmul结果，完成LeakyReLU 计算并暂存</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">由 Matmul API 管理结果交接，Vector 在结果就绪后执行激活</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px;">5</td>
    <td style="padding: 8px;">结果统一搬出：将LeakyReLU结果从LocalTensor搬至全局内存指定偏移地址</td>
    <td style="padding: 8px;">将计算结果放在正确的位置</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

### **2.4 融合核心优化效果**
- **算力利用率提升**：Cube/Vector 核在同一算子调用中协同工作，大幅减少核空闲时长，提升硬件利用率；


---
## **3. 基于高阶 API 的 CV 融合算子**

### **3.1 Host 侧 Tiling 实现**
Matmul在Cube上的数据流向为：GM →L1→L0A/L0B →Cube →L0C→FixPipe→GM  
LeakRelu在Vector上的数据流向为：GM → UB → Vector → UB → GM  
所以，在Host侧实现时，需要考虑每一次计算的数据小块大小不能超出L0C的大小和UB的大小。这里以A2设备展示：  
 
<img src="./images/03.05/a2_frame.png" alt="a2_frame"  width="700px">  

Host 侧调用 `MultiCoreMatmulTiling` 生成 `AscendC::tiling::TCubeTiling` 结构体，并作为Kernel参数直接传入Device侧。本示例中 Host 侧 `numBlocks = 1`，Kernel 使用 `__mix__(1, 2)`，对应1个AIC与2个AIV；`SetDim(2)` 配置参与 Matmul 发起的AIV数量。

In [ ]:
%%writefile Sources/03.05/matmul_leakyrelu_custom.asc
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

using namespace matmul;

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            std::fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__,    \
                         __code);                                                                       \
            return 1;                                                                                   \
        }                                                                                               \
    } while (0)

AscendC::tiling::TCubeTiling GenerateTiling(platform_ascendc::PlatformAscendC* ascendcPlatform)
{
    constexpr int32_t M = 1024;
    constexpr int32_t N = 640;
    constexpr int32_t K = 256;
    constexpr int32_t baseM = 128;
    constexpr int32_t baseN = 128;

    matmul_tiling::MultiCoreMatmulTiling cubeTiling(*ascendcPlatform);
    cubeTiling.SetDim(2);
    cubeTiling.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_FLOAT16);
    cubeTiling.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_FLOAT16);
    cubeTiling.SetCType(matmul_tiling::TPosition::VECIN, matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_FLOAT);
    cubeTiling.SetBiasType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                           matmul_tiling::DataType::DT_FLOAT);
    cubeTiling.SetShape(M, N, K);
    cubeTiling.SetOrgShape(M, N, K);
    cubeTiling.SetFixSplit(baseM, baseN, -1);
    cubeTiling.SetBias(true);
    cubeTiling.SetTraverse(matmul_tiling::MatrixTraverse::FIRSTM);
    cubeTiling.SetBufferSpace(-1, -1, -1);

    AscendC::tiling::TCubeTiling cubeTilingData;
    if (cubeTiling.GetTiling(cubeTilingData) == -1) {
        std::fprintf(stderr, "Generate matmul tiling failed.\n");
    }
    return cubeTilingData;
}

__aicore__ inline uint32_t Ceiling(uint32_t a, uint32_t b)
{
    return (a + b - 1) / b;
}



---
### **3.2 Kernel 侧代码实现**
`matmul_leakyrelu_custom.asc` 中完成CV融合算子的Kernel实现开发。
MatmulLeakyRelu融合算子的整体计算流程为：Matmul计算数据搬入->Matmul计算->Matmul计算结果搬出->LeakyRelu计算数据搬入->LeakyRelu计算->LeakyRelu计算结果搬出。  

<img src="./images/03.05/data_trends.png"  alt="data_trends"  width="700px">    

其中Matmul计算数据搬入、Matmul计算结果搬出和LeakyRelu计算数据搬入的操作可以通过Matmul API直接实现。 所以我们可以定义算子类包含以下函数：  
- MatmulCompute : 实现Matmul计算逻辑。
- LeakyReluCompute : 实现LeakyRelu计算逻辑。
- CopyOut : LeakyRelu计算结果搬出操作。CopyOut函数的实现逻辑与Vector算子运算结果的搬出流程不同。核心原因在于MatmulLeakyRelu算子的输出是一小块矩阵，导致其向GM的数据写入无法采用直接填充的方式。具体操作要求为：每完成一行小矩阵计算结果的填充后，需先跳过(整体矩阵单行元素数 - 小矩阵单行元素数)的地址偏移量，再执行下一行小矩阵结果的填充操作。    
    <img src="./images/03.05/result.png"  alt="result"  width="450px">   
   
- CalcOffset: 计算偏移函数。    
    1. **分块方式**：左矩阵（A）按行维度分块，右矩阵（B）按列维度分块，单个核负责的计算任务为「左矩阵某行块 × 右矩阵某列块」，运算结果对应结果矩阵（C）中对应的子矩阵区域。  

    2. **核任务分配逻辑（以示例说明）**：  
        若左矩阵分为 A1、A2 两个行块，右矩阵分为 B1、B2 两个列块，共启用 4 个核运算，则按 M 方向遍历规则分配任务：先按核编号依次遍历左矩阵所有行块，完成一轮遍历后，右矩阵列块向右滑动一列进入下一轮。具体分配为：  
        Core0：A1 × B1  
        Core1：A2 × B1  
        Core2：A1 × B2  
        Core3：A2 × B2   
        <img src="./images/03.05/offset.png"  alt="offset"  width="400px">     

    3. **分块索引计算**：  
    当前核处理的左矩阵行块 = 核编号 % 左矩阵总块数  
    当前核处理的右矩阵列块 = 核编号 / 左矩阵总块数（整除）  
    A 偏移 = 左块索引 × 单块行数 × A 列数（转置则为左块索引 × 单块行数）  
    B 偏移 = 右块索引 × 单块列数（转置则为右块索引 × 单块列数 × B 行数）  
    C 偏移 = 左块起始行 × C 总列数 + 右块起始列  

- Init：根据CalcOffset函数计算得到当前核处理的A矩阵偏移、B矩阵偏移、C矩阵偏移。  
- Process：设置当前核处理的A矩阵、B矩阵、C矩阵指针，并通过Iterate获取每次运算得到的C矩阵小块结果，用于LeakyRelu计算。

下面分 6 步追加 Kernel 代码。请先执行第 3.1 节创建源码的单元，再依次执行以下步骤；重新生成时也从第 3.1 节开始，避免重复追加。


#### **步骤 1：定义算子类与 Matmul 对象**

声明初始化、计算、搬出及偏移计算方法。`matmulObj` 指定 A、B、Bias 来自 GM，C 通过 `VECIN` 类型的 LocalTensor 接收；`reluOutQueue` 管理后续激活输出的片上缓冲区。


In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc

template <typename AType, typename BType, typename CType, typename BiasType>
class MatmulLeakyKernel {
public:
    __aicore__ inline void Init(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
                                const TCubeTiling& tiling, float alpha, AscendC::TPipe* pipe);
    __aicore__ inline void Process();
    __aicore__ inline void MatmulCompute();
    __aicore__ inline void LeakyReluCompute();
    __aicore__ inline void CopyOut(uint32_t count);
    __aicore__ inline void CalcOffset(int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA,
                                      int32_t& offsetB, int32_t& offsetC, int32_t& offsetBias);

    Matmul<MatmulType<AscendC::TPosition::GM, CubeFormat::ND, AType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BType>,
           MatmulType<AscendC::TPosition::VECIN, CubeFormat::ND, CType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BiasType>>
        matmulObj;

    AscendC::GlobalTensor<AType> aGlobal;
    AscendC::GlobalTensor<BType> bGlobal;
    AscendC::GlobalTensor<CType> cGlobal;
    AscendC::GlobalTensor<BiasType> biasGlobal;
    AscendC::LocalTensor<CType> reluOutLocal;
    TCubeTiling tiling;
    float alpha = 0.001f;
    AscendC::TQue<AscendC::QuePosition::VECOUT, 1> reluOutQueue;
};



#### **步骤 2：初始化数据地址与输出队列**

保存 Tiling 和激活斜率，将输入输出绑定为 `GlobalTensor`。`CalcOffset` 计算当前任务负责的矩阵块起点，随后将各 GM Tensor 移到该起点。最后通过 `InitBuffer` 分配一块 `baseM × baseN` 的输出缓冲区。


In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc
template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::Init(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c, const TCubeTiling& tiling,
    float alpha, AscendC::TPipe* pipe)
{
    this->tiling = tiling;
    this->alpha = alpha;
    aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ AType*>(a), tiling.M * tiling.Ka);
    bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ BType*>(b), tiling.Kb * tiling.N);
    cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ CType*>(c), tiling.M * tiling.N);
    biasGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ BiasType*>(bias), tiling.N);

    int32_t offsetA = 0;
    int32_t offsetB = 0;
    int32_t offsetC = 0;
    int32_t offsetBias = 0;
    CalcOffset(AscendC::GetBlockIdx(), tiling, offsetA, offsetB, offsetC, offsetBias);
    aGlobal = aGlobal[offsetA];
    bGlobal = bGlobal[offsetB];
    cGlobal = cGlobal[offsetC];
    biasGlobal = biasGlobal[offsetBias];
    pipe->InitBuffer(reluOutQueue, 1, tiling.baseM * tiling.baseN * sizeof(CType));
}



#### **步骤 3：逐块获取 Matmul 结果并执行激活计算**

`Process` 先设置 A、B 和 Bias，再通过 `Iterate<true>()` 逐块迭代。每轮依次获取 Matmul 结果、执行 LeakyReLU、搬出结果，完成后调用 `End()`。

`MatmulCompute` 申请 LocalTensor，并通过 `GetTensorC<true>` 获取当前结果；`LeakyReluCompute` 在该 Tensor 上原地执行激活计算，再将其入队交给搬出阶段。矩阵搬运与结果交接由高阶 Matmul API 管理。


In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc
template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::Process()
{
    uint32_t computeRound = 0;
    matmulObj.SetTensorA(aGlobal);
    matmulObj.SetTensorB(bGlobal);
    matmulObj.SetBias(biasGlobal);
    while (matmulObj.template Iterate<true>()) {
        MatmulCompute();
        LeakyReluCompute();
        CopyOut(computeRound);
        ++computeRound;
    }
    matmulObj.End();
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::MatmulCompute()
{
    reluOutLocal = reluOutQueue.AllocTensor<CType>();
    matmulObj.template GetTensorC<true>(reluOutLocal, false, true);
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::LeakyReluCompute()
{
    AscendC::LeakyRelu(reluOutLocal, reluOutLocal, static_cast<CType>(alpha), tiling.baseM * tiling.baseN);
    reluOutQueue.EnQue(reluOutLocal);
}



#### **步骤 4：按矩阵行跨度搬出激活结果**

`CopyOut` 从队列取出 Tensor，根据迭代序号计算当前 tile 在 C 中的起点。由于 tile 的行宽 `baseN` 可能小于完整矩阵的 N，每搬出一行需要跳过 `N-baseN` 个元素；此处 `DataCopyParams` 的长度和跨度按 32 字节单位填写。搬出后释放 Tensor。


In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc
template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::CopyOut(uint32_t count)
{
    reluOutQueue.DeQue<CType>();
    const uint32_t roundM = tiling.singleCoreM / tiling.baseM;
    uint32_t startOffset = count % roundM * tiling.baseM * tiling.N + count / roundM * tiling.baseN;
    AscendC::DataCopyParams copyParam = {
        static_cast<uint16_t>(tiling.baseM),
        static_cast<uint16_t>(tiling.baseN * sizeof(CType) / AscendC::DEFAULT_C0_SIZE),
        0,
        static_cast<uint16_t>((tiling.N - tiling.baseN) * sizeof(CType) / AscendC::DEFAULT_C0_SIZE)};
    AscendC::DataCopy(cGlobal[startOffset], reluOutLocal, copyParam);
    reluOutQueue.FreeTensor(reluOutLocal);
}



#### **步骤 5：计算任务对应的矩阵块偏移**

按 M 方向优先的顺序，将 block 编号映射为 M、N 两个方向的块编号。A 的起点随 M 块变化，B 和 Bias 的起点随 N 块变化，C 则同时取决于两者。这里计算整块任务的起点，步骤 4 再计算任务内部每轮 tile 的偏移。


In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc
template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::CalcOffset(
    int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA, int32_t& offsetB, int32_t& offsetC,
    int32_t& offsetBias)
{
    uint32_t mSingleBlocks = Ceiling(tiling.M, tiling.singleCoreM);
    uint32_t mCoreIdx = blockIdx % mSingleBlocks;
    uint32_t nCoreIdx = blockIdx / mSingleBlocks;
    offsetA = mCoreIdx * tiling.Ka * tiling.singleCoreM;
    offsetB = nCoreIdx * tiling.singleCoreN;
    offsetC = mCoreIdx * tiling.N * tiling.singleCoreM + nCoreIdx * tiling.singleCoreN;
    offsetBias = nCoreIdx * tiling.singleCoreN;
}



#### **步骤 6：定义混合核入口并注册 Matmul 对象**

入口创建 `TPipe` 和算子对象，初始化后通过 `REGIST_MATMUL_OBJ` 注册 Matmul 对象、workspace 与 Tiling，再执行计算流程。`__kfc_workspace__` 标记高阶 API 使用的系统 workspace，`__mix__(1, 2)` 指定混合核配比。


In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc
// 通过高阶API开发的CV融合算子，需要通过__kfc_workspace__标识workspace，由编译器完成workspace初始化。 
extern "C" __global__ __mix__(1, 2) void matmul_leakyrelu_custom(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
    __gm__ uint8_t* __kfc_workspace__ workspace, AscendC::tiling::TCubeTiling tiling)
{
    AscendC::TPipe pipe;

    MatmulLeakyKernel<half, half, float, float> op;
    op.Init(a, b, bias, c, tiling, 0.001f, &pipe);
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), op.matmulObj, &op.tiling);
    op.Process();
}


---
### **3.3 编译与测试**
完成Host侧tiling、Kernel实现和Host调用代码后，在同一个 `.asc` 文件中追加 `main` 函数。这里以固定的 M = 1024，N = 640，K = 256 进行测试，a矩阵全1，b矩阵全2，bias全0.5。

In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc

int32_t main()
{
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();

    constexpr uint32_t M = 1024;
    constexpr uint32_t N = 640;
    constexpr uint32_t K = 256;
    constexpr uint32_t numBlocks = 1;
    constexpr size_t aFileSize = M * K * sizeof(aclFloat16);
    constexpr size_t bFileSize = K * N * sizeof(aclFloat16);
    constexpr size_t biasFileSize = N * sizeof(float);
    constexpr size_t cFileSize = M * N * sizeof(float);
    const size_t workspaceSize = static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
    auto tiling = GenerateTiling(ascendcPlatform);

    constexpr int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    uint8_t* inputADevice = nullptr;
    uint8_t* inputBDevice = nullptr;
    uint8_t* inputBiasDevice = nullptr;
    uint8_t* outputCDevice = nullptr;
    uint8_t* workspaceDevice = nullptr;
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputADevice), aFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBDevice), bFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBiasDevice), biasFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&outputCDevice), cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&workspaceDevice), workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    std::vector<aclFloat16> inputAHost(M * K, aclFloatToFloat16(1.0f));
    std::vector<aclFloat16> inputBHost(K * N, aclFloatToFloat16(2.0f));
    std::vector<float> inputBiasHost(N, 0.5f);
    std::vector<float> outputCHost(M * N, 0.0f);
    std::vector<float> golden(M * N, 512.5f);

    CHECK_ACL(aclrtMemcpy(inputADevice, aFileSize, inputAHost.data(), aFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDevice, bFileSize, inputBHost.data(), bFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDevice, biasFileSize, inputBiasHost.data(), biasFileSize,
                          ACL_MEMCPY_HOST_TO_DEVICE));

    matmul_leakyrelu_custom<<<numBlocks, 0, stream>>>(inputADevice, inputBDevice, inputBiasDevice, outputCDevice,
                                                       workspaceDevice, tiling);
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(outputCHost.data(), cFileSize, outputCDevice, cFileSize, ACL_MEMCPY_DEVICE_TO_HOST));

    std::printf("result is:\n");
    for (uint32_t i = 0; i < 10; ++i) {
        std::printf("%.1f ", outputCHost[i]);
    }
    std::printf("\ntest %s\n", std::equal(outputCHost.begin(), outputCHost.end(), golden.begin()) ? "pass" : "failed");

    CHECK_ACL(aclrtFree(inputADevice));
    CHECK_ACL(aclrtFree(inputBDevice));
    CHECK_ACL(aclrtFree(inputBiasDevice));
    CHECK_ACL(aclrtFree(outputCDevice));
    if (workspaceDevice != nullptr) {
        CHECK_ACL(aclrtFree(workspaceDevice));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}


完成全部 `.asc` 代码后，还需要配置CMake来调用Ascend C编译工具链。由于本样例在Host侧调用 `MultiCoreMatmulTiling` 生成Matmul Tiling，并且调用 `PlatformAscendCManager` 获取硬件平台信息，因此还需要额外链接 `tiling_api`、`register`、`platform` 等CANN库。此外通过 `TARGET_SRC` 变量指定待编译的 `.asc` 源文件。

In [ ]:
%%writefile Sources/03.05/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, cpu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")
set(TARGET_SRC "matmul_leakyrelu_custom.asc" CACHE STRING "ASC source file used to build demo")

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    ${TARGET_SRC}
)

target_link_libraries(demo PRIVATE
    tiling_api
    register
    platform
    m
    dl
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)


编译直调工程，检查 `.asc` 文件能生成可执行程序。

In [ ]:
!cd Sources/03.05 && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && \
make -j

运行测试程序，检查输出是否与预期一致。

In [ ]:
!cd Sources/03.05/build && ./demo

调用成功后，会有如下输出：
```
result is:
512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 
test pass
```

---
## **4. 基于基础 API 的 CV 融合算子**

### **4.1 计算过程与核间分工**

高阶 Matmul API 封装了矩阵数据的搬运和计算过程。使用基础 API 时，这些操作由开发者使用基础API完成：AIC 从 GM 读取矩阵，在 Cube 上计算矩阵乘，再把结果交给 AIV 执行激活计算。

$$
c = leakyrelu(a \times b + bias, alpha)
$$

其中，`a`、`b` 为 half 类型输入矩阵，`bias` 为 float 类型的列偏置向量，`c` 为 float 类型输出矩阵。`alpha` 为 LeakyReLU 的负半轴斜率，本例取 `0.001`。对 `a × b + bias` 中的每个元素，非负值保持不变，负值乘以 `alpha`。

本节使用 `M = 512`、`K = 128`、`N = 128`，因此 `a`、`b`、`bias`、`c` 的形状分别为 `(512, 128)`、`(128, 128)`、`(128)`、`(512, 128)`。`bias` 沿 M 维广播，每一行使用同一组列偏置。

矩阵乘在 L0C 中使用 float 累加，Fixpipe 保持 float 类型写回 GM；AIV 读取结果，先加上 `bias`，再执行 LeakyReLU，得到最终输出 `c`。

输出矩阵沿 M 维分为两块，每块包含 `256×128` 个元素，由一个 AIC 完成矩阵乘。K 维长度为 128，每次矩阵乘覆盖完整的 K 维。一个 AIC 对应两个 AIV，因此每个 AIV 只需处理该块的一半，即 `128×128` 个元素。

`__mix__(1, 2)` 表示每个混合核组由 1 个 AIC 和 2 个 AIV 组成。当 Host 使用 `numBlocks=2` 发起计算时，共有 2 个 AIC 和 4 个 AIV 参与：

| AIC 编号 | 对应 AIV 编号 | AIV 分别处理的输出行 |
| --- | --- | --- |
| 0 | 0、1 | 0～127、128～255 |
| 1 | 2、3 | 256～383、384～511 |


### **4.2 静态 Tensor 与数据流**

矩阵乘和向量计算使用不同的片上存储。AIC 的输入经过 L1 后进入 L0A、L0B，累加结果保存在 L0C；AIV 则在 UB 中读取和更新向量数据。

这些数据分别由 `LocalTensor` 表示。`LocalMemAllocator<Hardware::...>` 指定分配的存储位置，`Alloc<T>(元素个数)` 返回对应类型和大小的 Tensor。本例中的元素个数由编译期模板参数确定，分配器按申请顺序安排地址，这就是这里采用的静态 Tensor 编程方式。

| 片上存储 | Tensor | 占用空间 |
| --- | --- | --- |
| L1 | a 的 `256×128` half + b 的 `128×128` half | 64 KiB + 32 KiB |
| L0A | a 的 `256×128` half | 64 KiB |
| L0B | b 的 `128×128` half | 32 KiB |
| L0C | c 的 `256×128` float | 128 KiB |
| UB（每个 AIV） | 半块 c 的 `128×128` float + bias 的 `128` float | 64 KiB + 512 b |

L1、L0A、L0B、L0C 和 UB 各自具有独立的容量限制。AIC 分支分配矩阵计算所需的 L1/L0 Tensor，AIV 分支分配激活所需的 UB Tensor。Tensor 的分配只确定数据存放的位置，搬运和计算之间的依赖仍由同步指令建立。

整个数据流如下：

<img src="./images/03.05/cv_basic_api_dataflow.png" alt="静态 Tensor 与基础 API 数据流：AIC 经 L1、L0A/L0B、L0C 完成矩阵乘并写回 GM，通过核间通知唤醒两个 AIV，分别对上、下半块加 bias 并激活后写回 c。" width="1200">

### **4.3 核内流水同步与核间同步**

AIC 写出矩阵乘结果后，AIV 才能读取这些结果。这个先后关系不会由源码中两个分支的排列顺序建立，因为 AIC 和 AIV 各自独立执行。`__mix__(1, 2)` 规定的是核的配比，数据何时可以被另一个核使用，还需要核间同步来通知。

本例通过 `CrossCoreSetFlag` 发送通知，通过 `CrossCoreWaitFlag` 等待通知。下面展示 AIC 写回与 AIV 读取之间的关系，搬运参数在后面的实现中展开：

```cpp
// AIC：将矩阵乘结果写回 GM，再由 FIX 流水发送通知。
AscendC::Fixpipe(cGM, cLocal, params);
AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(0);

// AIV：等待结果就绪，再从 GM 读取自己负责的半块。
AscendC::CrossCoreWaitFlag<0x2>(0);
AscendC::DataCopyPad<float>(vecLocal, cGM[gmOffset], copyParams, padParams);
```

这里的 `0x2` 表示同组 AIC/AIV 之间的 mode 2 同步，参数 `0` 是 flag ID。AIC 发送一次通知，同组两个 AIV 各等待一次；其他核组使用各自的配对关系。发送和等待使用相同的 mode 与 flag ID，才能对应到同一次数据交接。

发送端选择 `PIPE_FIX`，是因为 c 的写回由 FIX 流水完成。通知排在该流水的写回操作之后，AIV 等到通知时，才可以读取已经写好的结果。如果只等待矩阵乘完成，结果可能还停留在 L0C 中，尚未写入 GM。

同一个核内部也有多条独立流水。例如，AIC 的 MTE2 负责 GM→L1 搬运，MTE1 负责 L1→L0 搬运；后者读取 L1 前，需要等待前者写入完成。这类依赖使用 `SetFlag/WaitFlag<HardEvent::...>`，与 AIC/AIV 之间的核间事件作用范围不同。

| 范围 | 数据依赖 | 同步事件 |
| --- | --- | --- |
| AIC 核内 | GM→L1 完成后才能 LoadData | `MTE2_MTE1` |
| AIC 核内 | L1→L0 完成后才能 Mmad | `MTE1_M` |
| AIC 核内 | Mmad 完成后才能 Fixpipe | `M_FIX` |
| AIC→AIV 核间 | Fixpipe 写 GM 完成后才能读取 | `CrossCoreSetFlag<0x2, PIPE_FIX>` / `CrossCoreWaitFlag<0x2>` |
| AIV 核内 | 矩阵乘结果和 bias 搬入 UB 后才能相加 | `MTE2_V` |
| AIV 核内 | Add 完成后才能执行 LeakyReLU | `PipeBarrier<PIPE_V>()` |
| AIV 核内 | 激活完成后才能写回 GM | `V_MTE3` |

`PipeBarrier<PIPE_ALL>()` 等待的是当前核内的流水，不能替代核间通知。Host 上的 `aclrtSynchronizeStream` 则等待整个 Kernel 执行结束，它解决的是 Host 何时可以读取最终结果的问题。

本例中，AIC 对每块 c 只写入一次，随后由 AIV 原地更新，所以只使用 AIC→AIV 的“结果就绪”通知。如果 AIC 循环复用同一 GM 缓冲区，下一次写入还依赖两个 AIV 对旧数据的访问全部结束，此时就需要反向的“消费完成”通知。由于 AIV 还会把激活结果写回该区域，消费完成应包含它的 MTE3 写回完成。


### **4.4 Kernel 实现**

`KernelMatmul` 将上述过程组织为三个函数：`Init` 确定各核处理的 GM 区域，`Process` 执行 AIC/AIV 各自的计算，`LeakyReluCompute` 完成 AIV 的半块 bias 加法和激活。

下面按执行过程逐步展开。代码单元依次写入同一个 `.asc` 文件：步骤 1 创建文件，后续单元追加类和函数的剩余部分，最后接上第 4.5 节的 Host 代码。重新执行时从步骤 1 开始，可避免重复追加。


In [ ]:
!mkdir -p Sources/03.05/basic_api

#### **步骤 1：参数与 GM 地址初始化**

模板参数描述完整矩阵尺寸、单个 AIC 的计算尺寸以及一次矩阵乘的 tile 尺寸。本例每个 AIC 只计算一个 tile，因此 `singleCoreM/K/N` 与 `baseM/K/N` 分别相等；K 维也由这一次计算完整覆盖。`static_assert` 将这些关系和对齐要求限定在编译期。

`Init` 先用 `SetGlobalBuffer` 将指针绑定到 a、b、bias、c 的 `GlobalTensor`。对于 AIC，编号每增加 1，a 的起点向后移动 `singleCoreM*K` 个元素，c 的起点向后移动 `singleCoreM*N` 个元素。b 在两个 AIC 之间共用，不需要偏移。bias 在各个 AIV 之间共用，每一行使用相同的列偏置。

AIV 的 `cGM` 则定位到所属 AIC 的整块输出起点。`GetBlockIdx()/2` 将 AIV 0、1 映射到块 0，将 AIV 2、3 映射到块 1；块内的半块偏移在激活函数中计算。


In [ ]:
%%writefile Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

/* !
 * \file matmul_leakyrelu_basic_api.asc
 * \brief 基于静态Tensor编程模式实现Matmul和LeakyRelu融合计算，展示Cube单元和Vector单元协同计算的编程模式。
 *        样例矩阵规格为M=512、K=128、N=128，按singleCoreM=256、singleCoreN=128分核后，使用2个Cube核和4个Vector核完成计算。
 *        硬件Cube:Vector核数比例为1:2，2个Cube核负责Matmul计算，每个Cube核产生的结果再由2个Vector核协同完成LeakyRelu激活计算。
 */

#include "acl/acl.h"
#include "kernel_operator.h"
#include <cmath>
#include <cstdio>
#include <vector>

// half type, cube block: [16, 16]
constexpr uint32_t CUBE_BLOCK = 16;

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t singleCoreM, uint32_t singleCoreK, uint32_t singleCoreN,
    uint32_t baseM, uint32_t baseK, uint32_t baseN>
class KernelMatmul {
public:
    static_assert(M % singleCoreM == 0 && N == singleCoreN);
    static_assert(singleCoreM == baseM && singleCoreK == baseK && singleCoreN == baseN);
    static_assert(K == baseK && baseM % 32 == 0 && baseK % 16 == 0 && baseN % 16 == 0);
    __aicore__ inline KernelMatmul() {}
    __aicore__ inline void Init(__gm__ uint8_t* xMatrix, __gm__ uint8_t* yMatrix, __gm__ uint8_t* bias, __gm__ uint8_t* zMatrix)
    {
        aGM.SetGlobalBuffer((__gm__ half*)xMatrix);
        bGM.SetGlobalBuffer((__gm__ half*)yMatrix);
        cGM.SetGlobalBuffer((__gm__ float*)zMatrix);
        biasGM.SetGlobalBuffer((__gm__ float*)bias);
        // Cube核按M方向分核，每个Cube核负责singleCoreM * N大小输出块对应的A输入和C输出。
        if ASCEND_IS_AIC {
            aGM = aGM[AscendC::GetBlockIdx() * singleCoreM * K];
            cGM = cGM[AscendC::GetBlockIdx() * singleCoreM * N];
        }
        if ASCEND_IS_AIV {
            // Vector核数是Cube核数的2倍，1个Cube核的计算结果由2个Vector核继续完成LeakyRelu。
            // 因此这里用GetBlockIdx() / 2映射回对应的Cube核输出块起始地址。
            cGM = cGM[AscendC::GetBlockIdx() / 2 * singleCoreM * N];
        }
    }



#### **步骤 2：AIC 的静态 Tensor 分配**

`Process()` 通过 `ASCEND_IS_AIC` 选择 Cube 侧执行的代码。这里的四个分配器分别对应 L1、L0A、L0B 和 L0C。

`a1Local`、`b1Local` 保存搬入 L1 的矩阵，二者由同一个 L1 分配器依次分配。`a2Local`、`b2Local` 保存转换为 Cube 所需布局后的输入，分别位于 L0A、L0B。`cLocal` 位于 L0C，使用 float 类型保存矩阵乘累加结果。

这些 Tensor 此时还没有有效的输入数据；后面的搬运指令负责填充它们。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
    __aicore__ inline void Process()
    {
        if ASCEND_IS_AIC {
            // Matmul计算场景内存分配，需要在AIC侧执行。
            AscendC::LocalMemAllocator<AscendC::Hardware::L1> l1Allocator;
            AscendC::LocalMemAllocator<AscendC::Hardware::L0A> l0aAllocator;
            AscendC::LocalMemAllocator<AscendC::Hardware::L0B> l0bAllocator;
            AscendC::LocalMemAllocator<AscendC::Hardware::L0C> l0cAllocator;
            // 使用LocalMemAllocator按申请顺序分配片上内存，避免手动维护LocalTensor地址偏移。
            AscendC::LocalTensor<half> a1Local = l1Allocator.Alloc<half>(baseM * baseK);
            AscendC::LocalTensor<half> b1Local = l1Allocator.Alloc<half>(baseK * baseN);
            AscendC::LocalTensor<half> a2Local = l0aAllocator.Alloc<half>(baseM * baseK);
            AscendC::LocalTensor<half> b2Local = l0bAllocator.Alloc<half>(baseK * baseN);
            AscendC::LocalTensor<float> cLocal = l0cAllocator.Alloc<float>(baseM * baseN);


#### **步骤 3：将 a、b 从 GM 搬到 L1**

`DataCopy` 配合 `Nd2NzParams`，在搬运时将 GM 的 ND 矩阵转为 L1 的 NZ 布局。a 的行跨度为 K，b 的行跨度为 N；这两个跨度描述原始矩阵的行宽，而非搬运后的局部布局。

搬运由 MTE2 执行，下一步 `LoadData` 由 MTE1 执行，因此在两者之间设置并等待 `MTE2_MTE1` 事件。没有这组核内同步，MTE1 可能读取尚未搬完的 L1 数据。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
            // 本例 K 不切分，一次 Mmad 完成整个 K 维归约，产生 baseM*baseN 的结果
            // A矩阵GM->L1调用DataCopy接口 + Nd2NzParams结构体参数:
            // {ndNum, nValue, dValue, srcNdMatrixStride, srcDValue, dstNzC0Stride, dstNzNStride, dstNzMatrixStride}
            AscendC::DataCopy(a1Local, aGM, AscendC::Nd2NzParams{1, baseM, baseK, 0, K, baseM, 1, 0});
            // B矩阵GM->L1调用DataCopy接口 + Nd2NzParams结构体参数:
            // {ndNum, nValue, dValue, srcNdMatrixStride, srcDValue, dstNzC0Stride, dstNzNStride, dstNzMatrixStride}
            AscendC::DataCopy(b1Local, bGM, AscendC::Nd2NzParams{1, baseK, baseN, 0, N, baseK, 1, 0});

            // GM->L1的DataCopy由MTE2流水完成，后续L1->L0的LoadData由MTE1流水读取L1数据；
            // 这里设置MTE2_MTE1事件，使MTE1等待MTE2，避免LoadData读取到尚未搬运完成的数据。
            AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);
            AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);



#### **步骤 4：将 L1 数据搬到 L0A/L0B**

在 dav-2201 上，a 通过 `LoadData2DParams` 转成 L0A 的 ZZ 布局，b 转成 L0B 的 ZN 布局。两个循环都按 16×16 小块组织搬运：a 沿 M 方向遍历，b 沿 K 方向遍历；b 的 `ifTranspose=true` 用于形成矩阵乘需要的存储排布，不改变数学表达式 `a × b`。

代码中的 `512 / sizeof(half)` 等于 256 个元素，即一个 16×16 half 小块。搬运结束后通过 `MTE1_M` 事件，让矩阵计算流水等待 L0A/L0B 就绪。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
            // Nz->Zz, L0A上的排布格式为Zz
            // A矩阵L1->L0A调用LoadData接口 + LoadData2DParams结构体参数:
            // {startIndex, repeatTimes, srcStride, sid, dstGap, ifTranspose, addrMode}
            for (int i = 0; i < baseM / CUBE_BLOCK; ++i) {
                AscendC::LoadData(
                    a2Local[i * baseK * CUBE_BLOCK], a1Local[i * 512 / sizeof(half)],
                    AscendC::LoadData2DParams{0, baseK / CUBE_BLOCK, baseM / CUBE_BLOCK, 0, 0, false, 0});
            }
            // B矩阵L1->L0B（Nz->Zn）
            // B矩阵L1->L0B调用LoadData接口 + LoadData2DParams结构体参数:
            // {startIndex, repeatTimes, srcStride, sid, dstGap, ifTranspose, addrMode}
            for (int i = 0; i < baseK / CUBE_BLOCK; ++i) {
                AscendC::LoadData(
                    b2Local[i * baseN * CUBE_BLOCK], b1Local[i * 512 / sizeof(half)],
                    AscendC::LoadData2DParams{0, baseN / CUBE_BLOCK, baseK / CUBE_BLOCK, 0, 0, true, 0});
            }

            // L1->L0的LoadData由MTE1流水完成，后续Mmad由M流水读取L0A/L0B数据；
            // 这里设置MTE1_M事件，使M流水等待MTE1，避免Mmad读取到尚未搬运完成的数据。
            AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID0);
            AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID0);



#### **步骤 5：执行 Mmad，得到 L0C 累加结果**

调用 `Mmad(cLocal, a2Local, b2Local, ...)` 完成 `baseM × baseK` 与 `baseK × baseN` 的矩阵乘。`MmadParams` 的尺寸顺序为 M、N、K；最后的 `true` 表示初始化 c 累加矩阵。本例覆盖全部 K，只有一次 Mmad。

此时结果仍在 L0C 中，AIV 不能直接读取。先用 `M_FIX` 事件保证矩阵乘完成，再交给 FIX 流水搬出。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
            // 矩阵乘调用Mmad接口 + MmadParams结构体参数:{m, n, k, unitFlag, cmatrixSource, cmatrixInitVal}
            // 这里完成一个baseM * baseN大小tile的Matmul计算，结果保存在L0C中。
            AscendC::Mmad(cLocal, a2Local, b2Local, AscendC::MmadParams{baseM, baseN, baseK, 0, false, true});

            // Fixpipe由FIX流水读取L0C并搬运到GM；
            // 这里设置M_FIX事件，使FIX流水等待M流水，避免Fixpipe读取到尚未计算完成的数据。
            AscendC::SetFlag<AscendC::HardEvent::M_FIX>(EVENT_ID0);
            AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(EVENT_ID0);


#### **步骤 6：Fixpipe 写回 GM，并通知 AIV**

`Fixpipe` 将 L0C 的结果转为 ND，按行跨度 N 写入当前 AIC 的 c 块；`NoQuant` 保持 float 类型，不进行量化转换。

紧接着执行 `CrossCoreSetFlag<0x2, PIPE_FIX>(0)`：在生产结果的 FIX 流水上发送 flag 0，通知同组的两个 AIV。这里必须保证“结果写回完成”先于“消费者读取”，仅有前一步的 `M_FIX` 核内事件还不够。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
            // L0C->GM搬运调用Fixpipe接口 + FixpipeParamsV220结构体参数:
            // {nSize, mSize, srcStride, dstStride, reluEn, quantPre, deqScalar, ndNum,
            //  srcNdStride, dstNdStride, unitFlag}
            // 搬运时同时完成Nz到ND的格式转换，并保持float类型。
            AscendC::Fixpipe(
                cGM, cLocal,
                AscendC::FixpipeParamsV220{baseN, baseM, baseM, N, false, QuantMode_t::NoQuant, 0, 1, 0, 0, 0});
            // Cube核完成当前tile写回后，通过核间同步通知对应的2个Vector核可以开始读取该tile结果。
            AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(0);
        }


#### **步骤 7：AIV 的 UB 分配与核间等待**

`ASCEND_IS_AIV` 分支在 Vector 核上执行。每个 AIV 的 `vecLocal` 位于 UB，容纳 `baseM/2 × baseN` 个 float 元素，正好对应一个半块。`biasLocal` 另外保存 `baseN` 个 float 偏置元素。

UB 分配本身不读取矩阵乘结果，因此可以在 AIC 计算期间完成。`CrossCoreWaitFlag<0x2>(0)` 才是数据读取前的等待点：它与 AIC 的 `CrossCoreSetFlag<0x2, PIPE_FIX>(0)` 配对，将“Fixpipe 已写回”的条件传递给 AIV。

等待结束后，`LeakyReluCompute` 读取并处理当前 AIV 对应的半块。该函数中的地址计算和激活过程如下。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
        if ASCEND_IS_AIV {
            // Vector计算场景内存分配，需要在AIV侧执行。
            AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
            AscendC::LocalTensor<float> vecLocal = ubAllocator.Alloc<float>(baseM / 2 * baseN);
            AscendC::LocalTensor<float> biasLocal = ubAllocator.Alloc<float>(baseN);
            // Vector核需要先等待对应Cube核完成Fixpipe写回，再开始读取当前baseM * baseN tile的结果。
            AscendC::CrossCoreWaitFlag<0x2>(0);
            // 向量计算逻辑中，Cube:Vector=1:2，每个Vector核处理baseM / 2 * baseN大小的半块结果。
            LeakyReluCompute(vecLocal, biasLocal, 0, 0, baseM / 2 * baseN);
        }
    }



#### **步骤 8：计算半块偏移，并从 GM 搬入 UB**

`LeakyReluCompute` 中，`GetBlockIdx()%2` 决定当前 AIV 处理上半块还是下半块。注意 `cGM` 已在 `Init` 中加过所属 AIC 的整块偏移，这里只需加块内偏移，不能重复加整块偏移。

本例调用时 `mIndex=nIndex=0`，半块起始偏移因此为 `0` 或 `baseM/2 × N`。`DataCopyExtParams` 的 `blockCount` 表示搬运行数，`blockLen` 和 `srcStride` 以字节为单位；本例 `baseN=N`，行间跳过的字节数为 0。

矩阵乘结果搬入后，还通过 `DataCopy` 将对应列的 bias 搬入 `biasLocal`。本例每块覆盖全部 N 列，因此每个 AIV 读取完整的 bias 向量。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
private:
    __aicore__ inline void LeakyReluCompute(
        const AscendC::LocalTensor<float>& vecLocal,
        const AscendC::LocalTensor<float>& biasLocal, uint32_t mIndex, uint32_t nIndex, uint32_t tileLen)
    {
        uint32_t blockLen = baseN * sizeof(float);
        uint32_t srcStride = (N - baseN) * sizeof(float);

        // 每个Cube核产生的计算结果，由对应的2个Vector核完成计算，此处需要完成这2个Vector核的起始地址偏移计算
        uint32_t gmOffset = (mIndex * baseM * N + nIndex * baseN) + AscendC::GetBlockIdx() % 2 * (baseM / 2 * N);

        // 先把当前Cube tile中的一半结果从GM搬到UB，供当前Vector核做激活计算。
        // GM->UB调用DataCopyPad接口 + DataCopyExtParams结构体参数:{blockCount, blockLen, srcStride, dstStride, rsv}
        // DataCopyPadExtParams结构体参数:{isPad, leftPadding, rightPadding, paddingValue}
        AscendC::DataCopyPad<float>(
            vecLocal, cGM[gmOffset],
            AscendC::DataCopyExtParams{static_cast<uint16_t>(baseM / 2), blockLen, srcStride, 0, 0},
            AscendC::DataCopyPadExtParams<float>{true, 0, 0, 0});
        // Bias 按列广播：每一行加上同一个长度为 baseN 的向量。
        AscendC::DataCopy(biasLocal, biasGM[nIndex * baseN], baseN);



#### **步骤 9：加 bias、执行 LeakyReLU 并写回结果**

`MTE2_V` 保证矩阵乘结果和 bias 都已搬入 UB。随后逐行调用 `Add`，将同一个 bias 向量加到各行，实现 `a × b + bias`。`PipeBarrier<PIPE_V>()` 保证加法完成，再对 `vecLocal` 原地执行 float 类型的 `LeakyRelu`，负数乘以 `alpha`（代码中取 `0.001f`）。

随后用 `V_MTE3` 保证激活完成，再通过 `DataCopyPad` 将结果写回 c 的同一半块。两个 AIV 写入的区域互不重叠；AIC 也不会再次写这块 GM，因此本例不需要反向的“消费完成”通知。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
        // GM->UB的DataCopyPad由MTE2流水完成，后续LeakyRelu由V流水读取UB数据；
        // 这里设置MTE2_V事件，使V流水等待MTE2，避免向量计算读取到尚未搬运完成的数据。
        AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);

        // MTE2_V 同时保证矩阵乘结果和 Bias 已经搬入 UB。
        for (uint32_t row = 0; row < baseM / 2; ++row) {
            AscendC::Add(vecLocal[row * baseN], vecLocal[row * baseN], biasLocal, baseN);
        }
        // 加 Bias 和激活均使用 V 流水，确保 Add 的写入完成后再读取。
        AscendC::PipeBarrier<PIPE_V>();
        // 对 A*B+Bias 执行 LeakyRelu，负值部分乘以 0.001f。
        AscendC::LeakyRelu(vecLocal, vecLocal, 0.001f, tileLen);

        // LeakyRelu由V流水完成，后续DataCopyPad写回GM由MTE3流水执行；
        // 这里设置V_MTE3事件，使MTE3等待V流水，避免写回尚未计算完成的数据。
        AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);

        // 将当前Vector核负责的激活结果写回到原输出矩阵对应的半块区域。
        AscendC::DataCopyPad<float>(
            cGM[gmOffset], vecLocal,
            AscendC::DataCopyExtParams{static_cast<uint16_t>(baseM / 2), blockLen, 0, srcStride, 0});
    }



#### **步骤 10：成员变量与混合核入口**

类中的 `aGM`、`bGM`、`biasGM` 和 `cGM` 保存初始化后的全局 Tensor，供 `Process` 和 `LeakyReluCompute` 访问。

`mmad_vec_custom` 是 Host 发起调用的 Kernel 入口，`__mix__(1, 2)` 指定一个核组中的 AIC/AIV 配比。入口通过 `InitSocState()` 初始化运行状态，再创建 `KernelMatmul` 对象，依次执行地址初始化和计算。

末尾的 `PipeBarrier<PIPE_ALL>()` 等待核内流水收尾。AIV 何时开始读取 c，仍由前面的核间发送与等待控制。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc
private:
    AscendC::GlobalTensor<half> aGM;
    AscendC::GlobalTensor<half> bGM;
    AscendC::GlobalTensor<float> cGM;
    AscendC::GlobalTensor<float> biasGM;
};

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t singleCoreM, uint32_t singleCoreK, uint32_t singleCoreN,
    uint32_t baseM, uint32_t baseK, uint32_t baseN>
__global__ __mix__(1, 2) void mmad_vec_custom(__gm__ uint8_t* xMatrix, __gm__ uint8_t* yMatrix, __gm__ uint8_t* bias, __gm__ uint8_t* zMatrix)
{
    AscendC::InitSocState();
    KernelMatmul<M, K, N, singleCoreM, singleCoreK, singleCoreN, baseM, baseK, baseN> op;
    op.Init(xMatrix, yMatrix, bias, zMatrix);
    op.Process();
    AscendC::PipeBarrier<PIPE_ALL>();
}

### **4.5 Host 调用与精度校验**

矩阵尺寸和分块尺寸通过模板参数传给 Kernel，Host 因而不需要生成 `TCubeTiling`。`numBlocks=M/singleCoreM=2` 对应两个混合核组，设备内存分别保存 a、b、bias 和 c。

输入数据随行、列和 K 的索引变化，使矩阵乘结果同时包含正值、负值和零。这样既能检查各个 AIV 的地址偏移，也能覆盖 LeakyReLU 的负值计算分支。

bias 随列变化，包含正值、负值和零，用于检查列广播和加法。CPU 参考计算先完成矩阵乘，再按列加上 bias，最后使用 float 类型的 `alpha = 0.001f` 计算激活结果，与设备上的运算顺序一致。程序比较全部 `512×128=65536` 个输出，容差为 `1e-3 + 1e-3 × |expected|`；出现非有限值或误差超限时返回非零退出码。

程序固定使用设备 0。


In [ ]:
%%writefile -a Sources/03.05/basic_api/matmul_leakyrelu_basic_api.asc

#define CHECK_ACL(expr)                                                         \
    do {                                                                        \
        aclError ret = (expr);                                                   \
        if (ret != ACL_SUCCESS) {                                               \
            std::fprintf(stderr, "%s failed: %d\n", #expr, static_cast<int>(ret)); \
            return 1;                                                           \
        }                                                                       \
    } while (0)

int32_t main()
{
    constexpr uint32_t M = 512, K = 128, N = 128;
    constexpr uint32_t singleCoreM = 256, singleCoreK = 128, singleCoreN = 128;
    constexpr uint32_t baseM = 256, baseK = 128, baseN = 128;
    constexpr uint32_t numBlocks = M / singleCoreM; // 2 AIC + 4 AIV
    constexpr size_t aBytes = M * K * sizeof(aclFloat16);
    constexpr size_t bBytes = K * N * sizeof(aclFloat16);
    constexpr size_t cBytes = M * N * sizeof(float);
    constexpr size_t biasBytes = N * sizeof(float);
    constexpr int32_t deviceId = 0;

    // 整数输入可被 half 精确表示，数据随行、列、K 变化，覆盖所有分核边界。
    std::vector<aclFloat16> a(M * K), b(K * N);
    std::vector<float> c(M * N), bias(N);
    for (uint32_t j = 0; j < N; ++j) {
        bias[j] = (static_cast<int>(j % 5) - 2) * 0.25f;
    }
    for (uint32_t i = 0; i < M; ++i) {
        for (uint32_t k = 0; k < K; ++k) {
            a[i * K + k] = aclFloatToFloat16(static_cast<float>(
                (static_cast<int>(i % 7) - 3) * static_cast<int>(k % 3 + 1)));
        }
    }
    for (uint32_t k = 0; k < K; ++k) {
        for (uint32_t j = 0; j < N; ++j) {
            b[k * N + j] = aclFloatToFloat16(static_cast<float>(
                (static_cast<int>(j % 5) - 2) * static_cast<int>(k % 2 + 1)));
        }
    }
    aclrtStream stream = nullptr;
    uint8_t *aDevice = nullptr, *bDevice = nullptr, *cDevice = nullptr, *biasDevice = nullptr;
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&aDevice), aBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&bDevice), bBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&cDevice), cBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&biasDevice), biasBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMemcpy(aDevice, aBytes, a.data(), aBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(bDevice, bBytes, b.data(), bBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(biasDevice, biasBytes, bias.data(), biasBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    // 填充非结果值，便于发现漏写的行或半块。
    CHECK_ACL(aclrtMemset(cDevice, cBytes, 0x7f, cBytes));
    // 本例不调用高阶 Matmul，无需它的系统 workspace 和 Host Tiling。
    mmad_vec_custom<M, K, N, singleCoreM, singleCoreK, singleCoreN, baseM, baseK, baseN>
        <<<numBlocks, 0, stream>>>(aDevice, bDevice, biasDevice, cDevice);
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(c.data(), cBytes, cDevice, cBytes, ACL_MEMCPY_DEVICE_TO_HOST));

    size_t errors = 0, positive = 0, negative = 0, zero = 0;
    float maxError = 0.0f;
    constexpr float alpha = 0.001f;
    for (uint32_t i = 0; i < M; ++i) {
        for (uint32_t j = 0; j < N; ++j) {
            float sum = 0.0f;
            for (uint32_t k = 0; k < K; ++k) {
                sum += aclFloat16ToFloat(a[i * K + k]) * aclFloat16ToFloat(b[k * N + j]);
            }
            // 按列加 Bias，再执行 float LeakyReLU。
            float intermediate = sum + bias[j];
            float expected = intermediate >= 0.0f ? intermediate : intermediate * alpha;
            positive += expected > 0.0f;
            negative += expected < 0.0f;
            zero += expected == 0.0f;
            float actual = c[i * N + j];
            float error = std::fabs(actual - expected);
            if (error > maxError) maxError = error;
            if (!std::isfinite(actual) || error > 1e-3f + 1e-3f * std::fabs(expected)) {
                if (errors < 5) {
                    std::printf("mismatch [%u,%u]: actual=%g expected=%g\n", i, j, actual, expected);
                }
                ++errors;
            }
        }
    }
    const bool pass = errors == 0 && positive > 0 && negative > 0 && zero > 0;
    std::printf("checked=%zu, positive=%zu, negative=%zu, zero=%zu, errors=%zu, max_error=%g\n",
                c.size(), positive, negative, zero, errors, maxError);
    std::printf("basic API test %s\n", pass ? "pass" : "failed");
    CHECK_ACL(aclrtFree(aDevice));
    CHECK_ACL(aclrtFree(bDevice));
    CHECK_ACL(aclrtFree(cDevice));
    CHECK_ACL(aclrtFree(biasDevice));
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return pass ? 0 : 1;
}

### **4.6 编译与运行（dav-2201）**

`CMakeLists.txt` 使用 ASC 编译工具链构建 `.asc` 源码，目标架构为 `dav-2201`。该实现直接调用基础 API，不依赖高阶 Matmul 的 Host Tiling 库。`TARGET_SRC` 指定参与编译的源文件。

在第 1 节初始化环境、执行完本节源码单元后，以下命令完成编译和运行。


In [ ]:
%%writefile Sources/03.05/basic_api/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode")
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "A2/A3 architecture")
if(NOT CMAKE_ASC_ARCHITECTURES STREQUAL "dav-2201")
    message(FATAL_ERROR "This example supports dav-2201 only")
endif()
set(TARGET_SRC "matmul_leakyrelu_basic_api.asc" CACHE STRING "ASC source")
find_package(ASC REQUIRED)
project(cv_basic_api LANGUAGES ASC CXX)
add_executable(demo ${TARGET_SRC})
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

In [ ]:
!cmake -S Sources/03.05/basic_api -B Sources/03.05/basic_api/build -DCMAKE_ASC_ARCHITECTURES=dav-2201 && \
cmake --build Sources/03.05/basic_api/build -j && \
Sources/03.05/basic_api/build/demo

校验通过时，输出包含 `checked=65536`、`errors=0` 和 `basic API test pass`，分别表示比较的元素总数、超出容差的元素个数以及测试结果。

从执行过程可以看到，每个 AIV 都要等待对应 AIC 的整块矩阵乘结果。将单个 AIC 的任务进一步划分为多个 tile 后，AIV 就有机会在 AIC 计算下一块时处理上一块，从而形成 Cube 与 Vector 的流水重叠。如果多个 tile 复用同一缓冲区，“结果就绪”和“消费完成”两种通知共同决定该缓冲区何时可读、何时可重新写入。


---
## **5. 本章小结**

CV 融合将矩阵乘与逐元素计算放在同一个混合算子中：Cube 负责矩阵乘，Vector 负责 LeakyReLU 等激活操作。两类核通过中间结果建立生产与消费关系。

高阶 API 将矩阵数据搬运、计算和结果获取封装在 Matmul 对象中。开发者通过 Host Tiling 描述分块，在 Kernel 中逐块获取结果，再完成 Vector 计算和输出搬运。

基础 API 则显式展开这一过程：使用静态 Tensor 管理片上存储，通过 `DataCopy`、`LoadData`、`Mmad` 和 `Fixpipe` 完成矩阵乘的数据流，再由 AIV 将 GM 中的结果搬入 UB、执行激活并写回。

同步贯穿数据流的每一次交接。核内流水事件保证搬运与计算的先后关系，核间事件保证 AIV 在 AIC 写回完成后读取结果。分块后的偏移决定各核读写哪一段数据，同步决定这些数据何时可以使用；两者共同保证融合算子的正确性。


---
## **6. 课后实践**

实现 MatmulAbs 融合算子，完成矩阵乘后对结果逐元素取绝对值：

$$C = \operatorname{abs}(A \times B).$$

| 输入/输出 | 名称 | Shape | 数据类型 | 格式 |
| --- | --- | --- | --- | --- |
| 输入 | A | (512, 128) | half | ND |
| 输入 | B | (128, 128) | half | ND |
| 输出 | C | (512, 128) | float | ND |

本题不含 Bias。可自行选择高阶 API 或基础 API，在一个 CV 融合 Kernel 中完成矩阵乘与 Abs计算。分块、核函数参数、片上存储和流水同步由学习者自行实现。

### **编写算子**

在 `Sources/03.05-prac/matmul_abs.asc` 中完成实现。下方仅提供 Host 函数接口、测试数据和校验代码，AICore 代码及所需头文件自行添加。`kernel_matmul_abs` 接收 Host 输入，返回按行排列的 `M*N` 个 float 结果。


In [ ]:
!mkdir -p Sources/03.05-prac


In [ ]:
%%writefile Sources/03.05-prac/matmul_abs.asc
#include <cmath>
#include <cstdint>
#include <iostream>
#include <vector>

#include "acl/acl.h"

constexpr uint32_t M = 512;
constexpr uint32_t K = 128;
constexpr uint32_t N = 128;

// TODO: 自行添加所需头文件和 AICore 实现。

std::vector<float> kernel_matmul_abs(const std::vector<aclFloat16>& a,
                                     const std::vector<aclFloat16>& b)
{
    constexpr int32_t deviceId = 0;
    std::vector<float> output;

    // TODO: 完成 Host 调用流程并返回 M*N 个 float 结果。
    // 包括设备和 Stream 初始化、内存申请、输入拷贝、Kernel 调用、
    // 同步等待、结果拷回及资源释放；检查 ACL 调用的返回值。
    // Tiling、workspace、核函数参数和启动核数由所选实现自行决定。

    return output;
}

std::vector<float> ComputeGolden(const std::vector<aclFloat16>& a,
                                 const std::vector<aclFloat16>& b)
{
    std::vector<float> golden(M * N);
    for (uint32_t m = 0; m < M; ++m) {
        for (uint32_t n = 0; n < N; ++n) {
            float sum = 0.0f;
            for (uint32_t k = 0; k < K; ++k) {
                sum += aclFloat16ToFloat(a[m * K + k]) * aclFloat16ToFloat(b[k * N + n]);
            }
            golden[m * N + n] = std::fabs(sum);
        }
    }
    return golden;
}

int VerifyResult(const std::vector<float>& output, const std::vector<float>& golden)
{
    if (output.size() != golden.size()) {
        std::cerr << "[Failed] Output size: " << output.size()
                  << ", expected: " << golden.size() << '\n';
        return 1;
    }
    size_t errors = 0;
    for (size_t i = 0; i < golden.size(); ++i) {
        const float tolerance = 1e-3f + 1e-3f * std::fabs(golden[i]);
        if (!std::isfinite(output[i]) || std::fabs(output[i] - golden[i]) > tolerance) {
            if (errors < 5) {
                std::cerr << "Mismatch [" << i / N << ',' << i % N << "]: output="
                          << output[i] << ", golden=" << golden[i] << '\n';
            }
            ++errors;
        }
    }
    std::cout << "checked=" << golden.size() << ", errors=" << errors << '\n';
    std::cout << (errors == 0 ? "[Success] MatmulAbs verification passed.\n"
                             : "[Failed] MatmulAbs verification failed.\n");
    return errors == 0 ? 0 : 1;
}

int32_t main()
{
    std::vector<aclFloat16> a(M * K), b(K * N);
    // 输入随行、列及 K 变化，使矩阵乘结果包含正值、负值和零。
    for (uint32_t m = 0; m < M; ++m) {
        for (uint32_t k = 0; k < K; ++k) {
            const int value = (static_cast<int>(m % 7) - 3) * static_cast<int>(k % 3 + 1);
            a[m * K + k] = aclFloatToFloat16(static_cast<float>(value));
        }
    }
    for (uint32_t k = 0; k < K; ++k) {
        for (uint32_t n = 0; n < N; ++n) {
            const int value = (static_cast<int>(n % 5) - 2) * static_cast<int>(k % 2 + 1);
            b[k * N + n] = aclFloatToFloat16(static_cast<float>(value));
        }
    }
    const auto golden = ComputeGolden(a, b);
    const auto output = kernel_matmul_abs(a, b);
    return VerifyResult(output, golden);
}


### **编译与验证**

补全实现后，使用以下配置编译。若将实现拆分为多个源文件，可相应修改 `add_executable`。

测试数据使矩阵乘结果包含正值、负值和零。`ComputeGolden` 在 CPU 上计算 `abs(A×B)`，`VerifyResult` 比较全部输出元素，并检查输出长度和非有限值。未补全的 Host 函数返回空结果，验证会失败；完成后应输出 `checked=65536, errors=0` 和 `[Success] MatmulAbs verification passed.`。


In [ ]:
%%writefile Sources/03.05-prac/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode")
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture")
find_package(ASC REQUIRED)
project(matmul_abs_practice LANGUAGES ASC CXX)
add_executable(demo matmul_abs.asc)
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)
# 供选择高阶 API 时使用，基础 API 实现也可使用此配置。
target_link_libraries(demo PRIVATE tiling_api register platform m dl)


In [ ]:
!cmake -S Sources/03.05-prac -B Sources/03.05-prac/build -DCMAKE_ASC_ARCHITECTURES=dav-2201 && \
cmake --build Sources/03.05-prac/build -j && \
Sources/03.05-prac/build/demo
